# Rossmann Demand Forecasting: Larger Test Sample

This notebook extends the evaluation from 5 to 25 test stores, across the same three 28-day windows. The 5 original stores are kept, and 20 new stores are drawn at random from the remaining eligible stores.

The gradient boosting model is trained once per window on all eligible stores, as in the main pipeline, so the only change is the set of stores scored. ARIMA is optional, because it is slow: set `RUN_ARIMA = True` to include it.

Results are reported for the 20 new stores, for all 25 stores combined, and with a bootstrap confidence interval for the gap between the same-weekday mean and gradient boosting.

## 1. Setup

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

ROOT = Path(r"C:\Users\reinh\Downloads\Projects\Rossmann")
RAW_TRAIN = ROOT / "Rossmann" / "train.csv"
RAW_STORE = ROOT / "Rossmann" / "store.csv"
PROC = ROOT / "processed"
PROC.mkdir(parents=True, exist_ok=True)

HORIZON = 28
WINDOW_OFFSETS = [0, 28, 56]
ORIGINAL_TEST_STORES = [98, 480, 486, 735, 865]
N_NEW = 20
RUN_ARIMA = False        # set to True to include ARIMA (slow: about 60 fits)
BOOT_N = 5000

assert RAW_TRAIN.exists(), f"Not found: {RAW_TRAIN}"

## 2. Load data, eligibility and the test sample

In [2]:
train = pd.read_csv(RAW_TRAIN, dtype={"StateHoliday": str}, parse_dates=["Date"])
store = pd.read_csv(RAW_STORE)
calendar = pd.date_range(train["Date"].min(), train["Date"].max(), freq="D")

def longest_missing_run(dates):
    missing = calendar.difference(pd.DatetimeIndex(dates))
    if len(missing) == 0:
        return 0
    run = (pd.Series(missing).diff().dt.days != 1).cumsum()
    return int(run.value_counts().max())

coverage = train.groupby("Store")["Date"].nunique() / len(calendar)
longest_gap = train.groupby("Store")["Date"].apply(longest_missing_run)
stores = pd.DataFrame({"coverage": coverage, "longest_gap_days": longest_gap})
eligible = stores[(stores["coverage"] >= 0.95) & (stores["longest_gap_days"] < 14)]
print("Eligible stores:", len(eligible))

rng = np.random.default_rng(2026)
pool = [s for s in eligible.index.values if s not in ORIGINAL_TEST_STORES]
NEW_STORES = sorted(rng.choice(pool, size=N_NEW, replace=False).tolist())
ALL_TEST = sorted(ORIGINAL_TEST_STORES + NEW_STORES)

print("Original test stores:", ORIGINAL_TEST_STORES)
print("New test stores:", NEW_STORES)
print("Total test stores:", len(ALL_TEST))

Eligible stores: 935
Original test stores: [98, 480, 486, 735, 865]
New test stores: [28, 87, 110, 206, 207, 336, 390, 398, 405, 515, 707, 718, 733, 787, 808, 877, 926, 938, 959, 1005]
Total test stores: 25


## 3. Daily panel and features

Same construction as the main pipeline: full calendar, lags of at least 28 days, and calendar and store features.

In [3]:
STORES = eligible.index.values
panel = (
    train.set_index(["Store", "Date"])
    .reindex(pd.MultiIndex.from_product([STORES, calendar], names=["Store", "Date"]))
    .reset_index()
)
panel["DayOfWeek"] = panel["Date"].dt.dayofweek + 1
panel = panel.merge(store, on="Store", how="left")
panel = panel.sort_values(["Store", "Date"]).reset_index(drop=True)

g = panel.groupby("Store")["Sales"]
for k in [28, 35, 42, 49]:
    panel[f"lag_{k}"] = g.shift(k)
panel["same_dow_mean_4w"] = panel[["lag_28", "lag_35", "lag_42", "lag_49"]].mean(axis=1)
panel["roll_mean_28"] = g.transform(
    lambda s: s.shift(HORIZON).rolling(HORIZON, min_periods=14).mean()
)

panel["month"] = panel["Date"].dt.month
panel["day_of_month"] = panel["Date"].dt.day
panel["week_of_year"] = panel["Date"].dt.isocalendar().week.astype(int)
panel["StateHoliday"] = panel["StateHoliday"].fillna("0").map({"0": 0, "a": 1, "b": 2, "c": 3})
panel["StoreType"] = panel["StoreType"].astype("category").cat.codes
panel["Assortment"] = panel["Assortment"].astype("category").cat.codes
panel["CompetitorMissing"] = panel["CompetitionDistance"].isna().astype(int)
panel["CompetitionDistance"] = panel["CompetitionDistance"].fillna(100_000)

FEATURES = [
    "DayOfWeek", "Promo", "StateHoliday", "SchoolHoliday",
    "month", "day_of_month", "week_of_year",
    "StoreType", "Assortment", "CompetitionDistance", "CompetitorMissing", "Promo2",
    "lag_28", "lag_35", "lag_42", "lag_49", "same_dow_mean_4w", "roll_mean_28",
]
print("Panel rows:", f"{len(panel):,}")

Panel rows: 880,770


## 4. Model and baseline functions

In [4]:
def mape(actual, pred):
    return np.mean(np.abs(actual - pred) / actual) * 100

def wape(actual, pred):
    return np.sum(np.abs(actual - pred)) / np.sum(actual) * 100

def fit_model(cut):
    is_train = (panel["Date"] <= cut) & (panel["Open"] == 1) & panel["Sales"].notna()
    model = HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=100, early_stopping=False, random_state=42,
    )
    model.fit(panel.loc[is_train, FEATURES], np.log1p(panel.loc[is_train, "Sales"]))
    return model

def naive_forecast(cut, ho):
    parts = []
    for s, d in ho.groupby("Store"):
        hist = panel[(panel["Store"] == s) & (panel["Date"] <= cut)].tail(HORIZON)
        by_dow = hist.groupby("DayOfWeek")["Sales"].mean()
        parts.append(pd.Series(d["DayOfWeek"].map(by_dow).values, index=d.index))
    return pd.concat(parts)

def fill_for_fit(y):
    return y.interpolate(limit_direction="both")

def arima_for_store(s, cut, end):
    d = panel[panel["Store"] == s].set_index("Date")
    y_tr = d["Sales"].loc[:cut]
    y_te = d.loc[cut + pd.Timedelta(days=1):end]
    y = np.log1p(fill_for_fit(y_tr))
    model = SARIMAX(
        y, order=(1, 0, 1), seasonal_order=(1, 0, 1, 7),
        enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False)
    pred = np.expm1(model.forecast(HORIZON)).clip(lower=0).values
    return pred, y_te

## 5. Run the three windows on all 25 test stores

One model is fitted per window. The 28 days after each cutoff are scored on open days with positive sales.

In [5]:
rows = []
for off in WINDOW_OFFSETS:
    cut = calendar[-HORIZON - 1 - off]
    end = cut + pd.Timedelta(days=HORIZON)
    model = fit_model(cut)

    ho = panel[(panel["Date"] > cut) & (panel["Date"] <= end) & panel["Store"].isin(ALL_TEST)].copy()
    ho["pred_gb"] = np.expm1(model.predict(ho[FEATURES])).clip(min=0)
    ho["pred_naive"] = naive_forecast(cut, ho)
    ho = ho[(ho["Open"] == 1) & (ho["Sales"] > 0)]

    for s, d in ho.groupby("Store"):
        for name, col in [("Gradient boosting", "pred_gb"), ("Same-weekday mean", "pred_naive")]:
            rows.append({"cutoff": cut.date(), "Store": s, "Model": name,
                         "MAPE_%": mape(d["Sales"], d[col]), "WAPE_%": wape(d["Sales"], d[col])})

        if RUN_ARIMA:
            pred, y_te = arima_for_store(s, cut, end)
            keep = (y_te["Open"].eq(1) & y_te["Sales"].notna() & (y_te["Sales"] > 0)).values
            rows.append({"cutoff": cut.date(), "Store": s, "Model": "ARIMA with weekly seasonality",
                         "MAPE_%": mape(y_te["Sales"].values[keep], pred[keep]),
                         "WAPE_%": wape(y_te["Sales"].values[keep], pred[keep])})
    print("Done window", cut.date())

res = pd.DataFrame(rows)
res["group"] = np.where(res["Store"].isin(ORIGINAL_TEST_STORES), "original 5", "new 20")
print("Store-windows scored:", len(res) // len(res["Model"].unique()))

Done window 2015-07-03
Done window 2015-06-05
Done window 2015-05-08
Store-windows scored: 75


## 6. Results: all 25 stores, and the 20 new stores on their own

In [6]:
print("All 25 stores, average per window:")
print(res.groupby(["cutoff", "Model"])[["MAPE_%", "WAPE_%"]].mean().round(2))

print("\nAll 25 stores, average across windows:")
print(res.groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2).sort_values("MAPE_%"))

print("\nNew 20 stores only, average across windows:")
print(res[res["group"] == "new 20"].groupby("Model")[["MAPE_%", "WAPE_%"]].mean().round(2).sort_values("MAPE_%"))

print("\nSpread of window averages across all 25 stores (std of MAPE):")
print(res.groupby(["cutoff", "Model"])["MAPE_%"].mean().groupby("Model").std().round(2))

All 25 stores, average per window:
                              MAPE_%  WAPE_%
cutoff     Model                            
2015-05-08 Gradient boosting    9.28    9.30
           Same-weekday mean   17.21   17.11
2015-06-05 Gradient boosting    9.54    9.25
           Same-weekday mean   21.45   22.01
2015-07-03 Gradient boosting    8.27    7.99
           Same-weekday mean   16.83   16.20

All 25 stores, average across windows:
                   MAPE_%  WAPE_%
Model                            
Gradient boosting    9.03    8.84
Same-weekday mean   18.49   18.44

New 20 stores only, average across windows:
                   MAPE_%  WAPE_%
Model                            
Gradient boosting    8.99    8.80
Same-weekday mean   18.14   18.12

Spread of window averages across all 25 stores (std of MAPE):
Model
Gradient boosting    0.67
Same-weekday mean    2.57
Name: MAPE_%, dtype: float64


## 7. How certain is the gap?

Each store's MAPE is averaged over the three windows. The bootstrap then resamples stores with replacement, to estimate how much the gap between the same-weekday mean and gradient boosting could vary with a different set of stores.

In [7]:
per_store = res.groupby(["Store", "Model"])["MAPE_%"].mean().unstack()
gap = (per_store["Same-weekday mean"] - per_store["Gradient boosting"]).dropna()

print("Stores in the comparison:", len(gap))
print("Average gap (same-weekday minus gradient boosting), MAPE points:", round(gap.mean(), 2))
print("Stores where gradient boosting is better:", int((gap > 0).sum()), "of", len(gap))

boot_rng = np.random.default_rng(0)
boot_means = np.array([
    boot_rng.choice(gap.values, size=len(gap), replace=True).mean() for _ in range(BOOT_N)
])
lo, hi = np.percentile(boot_means, [2.5, 97.5])
print(f"95% bootstrap interval for the average gap: {lo:.2f} to {hi:.2f} MAPE points")
print("Interval entirely above zero:", lo > 0)

Stores in the comparison: 25
Average gap (same-weekday minus gradient boosting), MAPE points: 9.46
Stores where gradient boosting is better: 25 of 25
95% bootstrap interval for the average gap: 8.19 to 10.74 MAPE points
Interval entirely above zero: True


## 8. Save

In [8]:
SAMPLE_OUT = PROC / "sample_extension_results.csv"
res.round(2).to_csv(SAMPLE_OUT, index=False)
print("Saved:", SAMPLE_OUT)

Saved: C:\Users\reinh\Downloads\Projects\Rossmann\processed\sample_extension_results.csv
